# ENARES 2024 CRS04 — Stage 03

## Notebook 08 — Búsqueda de ayuda frente a violencia sexual

Traducción reproducible de `13_CRS04_3.6_BusquedaAyuda_VS_ver4.sps`.

Crea indicadores de búsqueda de ayuda informal, persona a quien pidió ayuda, ayuda recibida, brechas, apoyo institucional y DEMUNA.

- BigQuery Sandbox.
- `CREATE OR REPLACE TABLE`.
- 18,807 filas esperadas.
- Se preservan los `NULL` metodológicos definidos por SPSS.
- Los tabulados con muestras complejas se ejecutan después en R.

In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes

In [2]:
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

auth.authenticate_user()
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_ID = 'enares-2024-crs04'
LOCATION = 'US'
EXPECTED_ROWS = 18807

ROOT_DRIVE = Path('/content/drive/MyDrive/ENARES_2024_PROJECT')
LOG_DIR = ROOT_DRIVE / '05Resultados' / 'logs' / 'stage03'
SQL_DIR = ROOT_DRIVE / '02SQL'

for directory in [LOG_DIR, SQL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

A = f'{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents'

print('Target table:', A)
print('RUN_UTC:', RUN_UTC)

Target table: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents
RUN_UTC: 2026-07-17T05:11:40.746047+00:00


In [4]:
required_inputs = (
    ['VS_12M','C4P252','C4P254','C4P256','C4P257','C4P259','C3P246','C3P247']
    + [f'C4P253_{i}' for i in range(1,22)]
    + [f'C4P255_{i}' for i in range(1,8)]
    + [f'C4P258_{i}' for i in range(1,13)]
    + [f'C4P260_{i}' for i in range(1,5)]
)

table_before = client.get_table(A)
existing_columns = {field.name for field in table_before.schema}
missing_inputs = sorted(set(required_inputs) - existing_columns)

if missing_inputs:
    raise RuntimeError('No se puede ejecutar la sección 3.6. Faltan variables: ' + ', '.join(missing_inputs))
if table_before.num_rows != EXPECTED_ROWS:
    raise RuntimeError(f'La tabla tiene {table_before.num_rows:,} filas; se esperaban {EXPECTED_ROWS:,}.')

print('Prerequisitos aprobados.')
print('Filas:', table_before.num_rows)
print('Variables requeridas:', len(required_inputs))

Prerequisitos aprobados.
Filas: 18807
Variables requeridas: 52


In [5]:
domain_variables = ['VS_12M','C4P252','C4P254','C4P256','C4P257','C4P259','C3P246','C3P247']
parts=[]
for variable in domain_variables:
    parts.append(f"""
    SELECT '{variable}' AS variable, CAST(`{variable}` AS STRING) AS value, COUNT(*) AS n
    FROM `{A}`
    GROUP BY `{variable}`
    """)
source_domain_sql='\nUNION ALL\n'.join(parts)
source_domain=client.query(source_domain_sql, location=LOCATION).result().to_dataframe()
source_domain.to_csv(LOG_DIR/'stage3_36_busqueda_ayuda_source_domain.csv', index=False)
display(source_domain.sort_values(['variable','value'], na_position='first'))

,variable,value,n
23,C3P246,1,8776
24,C3P246,2,10031
26,C3P247,None,10031
27,C3P247,1,417
25,C3P247,2,8359
4,C4P252,None,13014
3,C4P252,1,2534
2,C4P252,2,3232
5,C4P252,3,27
6,C4P254,None,16273


In [6]:
outputs = [
'filtro_vs_12m','busco_ayuda_vs','ayuda_vs_familiar','ayuda_vs_madre','ayuda_vs_padre',
'ayuda_vs_madrastra','ayuda_vs_padrastro','ayuda_vs_hermana','ayuda_vs_hermano',
'ayuda_vs_abuela','ayuda_vs_abuelo','ayuda_vs_tia','ayuda_vs_tio','ayuda_vs_otro_pariente',
'ayuda_vs_car','ayuda_vs_escolar_adulto','ayuda_vs_pares_amigos','ayuda_vs_otro',
'recibio_ayuda_vs','recibio_ayuda_vs_victimas','brecha_ayuda_vs','ayuda_vs_consejo',
'ayuda_vs_hablo_madre_padre','ayuda_vs_reclamo_agresor','ayuda_vs_aviso_autoridades',
'ayuda_vs_refugio','ayuda_vs_especialista','ayuda_vs_otro_tipo','apoyo_institucional_vs',
'recibio_ayuda_institucional_vs','ayuda_inst_vs_hablo_familia','ayuda_inst_vs_terapias',
'ayuda_inst_vs_llamo_atencion','ayuda_inst_vs_otro','brecha_institucional_vs',
'conoce_demuna','uso_demuna']

columns_to_replace=[c for c in outputs if c in existing_columns]
source_select='* EXCEPT(' + ', '.join(f'`{c}`' for c in columns_to_replace) + ')' if columns_to_replace else '*'
print('Variables to create:', len(outputs))

Variables to create: 37


In [7]:
sql_busqueda_ayuda = f"""
CREATE OR REPLACE TABLE `{A}` AS
WITH base AS (
  SELECT {source_select} FROM `{A}`
),
derived AS (
  SELECT
    *,
    CASE WHEN VS_12M=1 THEN 1 ELSE 0 END AS filtro_vs_12m,
    CASE WHEN C4P252=1 THEN 1 WHEN C4P252=2 THEN 0 WHEN C4P252=3 THEN NULL ELSE NULL END AS busco_ayuda_vs,

    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN
      C4P253_1=1 OR C4P253_2=1 OR C4P253_3=1 OR C4P253_4=1 OR C4P253_5=1 OR
      C4P253_6=1 OR C4P253_7=1 OR C4P253_8=1 OR C4P253_9=1 OR C4P253_10=1 OR C4P253_11=1
      THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_familiar,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_1=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_madre,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_2=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_padre,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_3=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_madrastra,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_4=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_padrastro,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_5=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_hermana,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_6=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_hermano,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_7=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_abuela,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_8=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_abuelo,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_9=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_tia,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_10=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_tio,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_11=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_otro_pariente,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_12=1 OR C4P253_13=1 OR C4P253_14=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_car,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_15=1 OR C4P253_16=1 OR C4P253_17=1 OR C4P253_18=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_escolar_adulto,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_19=1 OR C4P253_20=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_pares_amigos,
    CASE WHEN C4P252 IN (1,2) THEN CASE WHEN C4P253_21=1 THEN 1 ELSE 0 END ELSE NULL END AS ayuda_vs_otro,

    CASE WHEN C4P254=1 THEN 1 WHEN C4P254=2 THEN 0 WHEN C4P254=3 THEN NULL ELSE NULL END AS recibio_ayuda_vs,
    CASE WHEN VS_12M!=1 OR VS_12M IS NULL THEN NULL WHEN C4P254=3 THEN NULL WHEN C4P254=1 THEN 1 ELSE 0 END AS recibio_ayuda_vs_victimas,

    CASE WHEN C4P255_1=1 THEN 1 ELSE 0 END AS ayuda_vs_consejo,
    CASE WHEN C4P255_2=1 THEN 1 ELSE 0 END AS ayuda_vs_hablo_madre_padre,
    CASE WHEN C4P255_3=1 THEN 1 ELSE 0 END AS ayuda_vs_reclamo_agresor,
    CASE WHEN C4P255_4=1 THEN 1 ELSE 0 END AS ayuda_vs_aviso_autoridades,
    CASE WHEN C4P255_5=1 THEN 1 ELSE 0 END AS ayuda_vs_refugio,
    CASE WHEN C4P255_6=1 THEN 1 ELSE 0 END AS ayuda_vs_especialista,
    CASE WHEN C4P255_7=1 THEN 1 ELSE 0 END AS ayuda_vs_otro_tipo,

    CASE WHEN C4P257=1 THEN 1 WHEN C4P257=2 THEN 0 ELSE NULL END AS apoyo_institucional_vs,
    CASE WHEN C4P259=1 THEN 1 WHEN C4P259=2 THEN 0 WHEN C4P259=3 THEN NULL ELSE NULL END AS recibio_ayuda_institucional_vs,
    CASE WHEN C4P260_1=1 THEN 1 ELSE 0 END AS ayuda_inst_vs_hablo_familia,
    CASE WHEN C4P260_2=1 THEN 1 ELSE 0 END AS ayuda_inst_vs_terapias,
    CASE WHEN C4P260_3=1 THEN 1 ELSE 0 END AS ayuda_inst_vs_llamo_atencion,
    CASE WHEN C4P260_4=1 THEN 1 ELSE 0 END AS ayuda_inst_vs_otro,

    CASE WHEN C3P246=1 THEN 1 WHEN C3P246=2 THEN 0 ELSE NULL END AS conoce_demuna,
    CASE WHEN C3P247=1 THEN 1 WHEN C3P247=2 THEN 0 ELSE NULL END AS uso_demuna
  FROM base
),
final AS (
  SELECT
    *,
    CASE WHEN busco_ayuda_vs=1 AND recibio_ayuda_vs=0 THEN 1
         WHEN busco_ayuda_vs=1 AND recibio_ayuda_vs=1 THEN 0 ELSE NULL END AS brecha_ayuda_vs,
    CASE WHEN VS_12M=1 AND apoyo_institucional_vs=0 THEN 1
         WHEN VS_12M=1 AND apoyo_institucional_vs=1 THEN 0 ELSE NULL END AS brecha_institucional_vs
  FROM derived
)
SELECT * FROM final
"""

sql_path = SQL_DIR / 'stage3_36_busqueda_ayuda_vs.sql'
sql_path.write_text(sql_busqueda_ayuda, encoding='utf-8')
client.query(sql_busqueda_ayuda, location=LOCATION).result()
print('Stage 3.6 variables created.')
print('SQL saved to:', sql_path)

Stage 3.6 variables created.
SQL saved to: /content/drive/MyDrive/ENARES_2024_PROJECT/02SQL/stage3_36_busqueda_ayuda_vs.sql


In [8]:
table_after=client.get_table(A)
columns_after={field.name for field in table_after.schema}
missing_outputs=sorted(set(outputs)-columns_after)
if missing_outputs:
    raise RuntimeError('La consulta terminó, pero faltan columnas: '+', '.join(missing_outputs))
if table_after.num_rows != EXPECTED_ROWS:
    raise RuntimeError(f'La transformación dejó {table_after.num_rows:,} filas; se esperaban {EXPECTED_ROWS:,}.')
print('Output columns confirmed:', len(outputs))
print('Rows preserved:', table_after.num_rows)

Output columns confirmed: 37
Rows preserved: 18807


In [9]:
parts=[]
for variable in outputs:
    parts.append(f"""
    SELECT '{variable}' AS variable, CAST(`{variable}` AS STRING) AS invalid_value, COUNT(*) AS n
    FROM `{A}`
    WHERE `{variable}` IS NOT NULL AND `{variable}` NOT IN (0,1)
    GROUP BY `{variable}`
    """)
invalid_domains_sql='\nUNION ALL\n'.join(parts)
invalid_domains=client.query(invalid_domains_sql, location=LOCATION).result().to_dataframe()

validation=client.query(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(busco_ayuda_vs=1 AND C4P252!=1) AS bad_busco_ayuda,
  COUNTIF(recibio_ayuda_vs=1 AND C4P254!=1) AS bad_recibio_ayuda,
  COUNTIF(recibio_ayuda_vs_victimas IS NOT NULL AND VS_12M!=1) AS received_outside_victim_universe,
  COUNTIF(brecha_ayuda_vs IS NOT NULL AND busco_ayuda_vs!=1) AS informal_gap_outside_denominator,
  COUNTIF(brecha_institucional_vs IS NOT NULL AND VS_12M!=1) AS institutional_gap_outside_denominator,
  COUNTIF(brecha_institucional_vs=1 AND apoyo_institucional_vs!=0) AS inconsistent_institutional_gap,
  COUNTIF(ayuda_vs_familiar=1 AND NOT (C4P253_1=1 OR C4P253_2=1 OR C4P253_3=1 OR C4P253_4=1 OR C4P253_5=1 OR C4P253_6=1 OR C4P253_7=1 OR C4P253_8=1 OR C4P253_9=1 OR C4P253_10=1 OR C4P253_11=1)) AS inconsistent_family_group,
  COUNTIF(ayuda_vs_car=1 AND NOT (C4P253_12=1 OR C4P253_13=1 OR C4P253_14=1)) AS inconsistent_car_group,
  COUNTIF(ayuda_vs_escolar_adulto=1 AND NOT (C4P253_15=1 OR C4P253_16=1 OR C4P253_17=1 OR C4P253_18=1)) AS inconsistent_school_adult_group
FROM `{A}`
""", location=LOCATION).result().to_dataframe()

display(invalid_domains)
display(validation)
invalid_domains.to_csv(LOG_DIR/'stage3_36_invalid_domains.csv', index=False)
validation.to_csv(LOG_DIR/'stage3_36_busqueda_ayuda_validation.csv', index=False)

if len(invalid_domains)>0:
    raise RuntimeError('Hay variables derivadas con valores fuera de 0/1/NULL.')
row=validation.iloc[0]
checks=[c for c in validation.columns if c!='total_rows']
failed={c:int(row[c]) for c in checks if int(row[c])>0}
if int(row['total_rows'])!=EXPECTED_ROWS:
    raise RuntimeError('Conteo de filas incorrecto.')
if failed:
    raise RuntimeError('Fallaron controles de calidad: '+', '.join(f'{k}={v}' for k,v in failed.items()))
print('QA approved.')

,variable,invalid_value,n


,total_rows,bad_busco_ayuda,bad_recibio_ayuda,received_outside_victim_universe,informal_gap_outside_denominator,institutional_gap_outside_denominator,inconsistent_institutional_gap,inconsistent_family_group,inconsistent_car_group,inconsistent_school_adult_group
0,18807,0,0,0,0,0,0,0,0,0


QA approved.


In [10]:
parts=[]
for variable in outputs:
    parts.append(f"""
    SELECT '{variable}' AS variable, CAST(`{variable}` AS STRING) AS value, COUNT(*) AS n,
           ROUND(100*SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER (PARTITION BY 1)),4) AS percent_total
    FROM `{A}`
    GROUP BY `{variable}`
    """)
distribution_sql='\nUNION ALL\n'.join(parts)
distributions=client.query(distribution_sql, location=LOCATION).result().to_dataframe()
distributions.to_csv(LOG_DIR/'stage3_36_busqueda_ayuda_distributions.csv', index=False)
display(distributions.sort_values(['variable','value'], na_position='first'))

,variable,value,n,percent_total
76,apoyo_institucional_vs,None,13014,69.1976
75,apoyo_institucional_vs,0,5490,29.1913
77,apoyo_institucional_vs,1,303,1.6111
81,ayuda_inst_vs_hablo_familia,0,18730,99.5906
82,ayuda_inst_vs_hablo_familia,1,77,0.4094
...,...,...,...,...
56,recibio_ayuda_vs_victimas,0,2101,11.1714
57,recibio_ayuda_vs_victimas,1,1323,7.0346
94,uso_demuna,None,10031,53.3365
95,uso_demuna,0,8359,44.4462


In [11]:
informal=client.query(f"""
SELECT VS_12M,busco_ayuda_vs,recibio_ayuda_vs,recibio_ayuda_vs_victimas,brecha_ayuda_vs,COUNT(*) AS n
FROM `{A}`
GROUP BY VS_12M,busco_ayuda_vs,recibio_ayuda_vs,recibio_ayuda_vs_victimas,brecha_ayuda_vs
ORDER BY VS_12M,busco_ayuda_vs,recibio_ayuda_vs
""", location=LOCATION).result().to_dataframe()

institutional=client.query(f"""
SELECT VS_12M,apoyo_institucional_vs,recibio_ayuda_institucional_vs,brecha_institucional_vs,COUNT(*) AS n
FROM `{A}`
GROUP BY VS_12M,apoyo_institucional_vs,recibio_ayuda_institucional_vs,brecha_institucional_vs
ORDER BY VS_12M,apoyo_institucional_vs,recibio_ayuda_institucional_vs
""", location=LOCATION).result().to_dataframe()

informal.to_csv(LOG_DIR/'stage3_36_informal_help_crosstab.csv', index=False)
institutional.to_csv(LOG_DIR/'stage3_36_institutional_help_crosstab.csv', index=False)
display(informal)
display(institutional)

,VS_12M,busco_ayuda_vs,recibio_ayuda_vs,recibio_ayuda_vs_victimas,brecha_ayuda_vs,n
0,0,<NA>,<NA>,<NA>,<NA>,13022
1,0,0,<NA>,<NA>,<NA>,1244
2,0,1,<NA>,<NA>,<NA>,4
3,0,1,0,<NA>,1,96
4,0,1,1,<NA>,0,1012
5,1,<NA>,<NA>,0,<NA>,19
6,1,0,<NA>,0,<NA>,1988
7,1,1,<NA>,<NA>,<NA>,5
8,1,1,0,0,1,94
9,1,1,1,1,0,1323


,VS_12M,apoyo_institucional_vs,recibio_ayuda_institucional_vs,brecha_institucional_vs,n
0,0,<NA>,<NA>,<NA>,13014
1,0,0,<NA>,<NA>,2228
2,0,1,<NA>,<NA>,1
3,0,1,0,<NA>,19
4,0,1,1,<NA>,116
5,1,0,<NA>,1,3262
6,1,1,<NA>,0,6
7,1,1,0,0,29
8,1,1,1,0,132


In [12]:
closure=pd.DataFrame([{
    'run_utc':RUN_UTC,
    'table':A,
    'expected_rows':EXPECTED_ROWS,
    'actual_rows':table_after.num_rows,
    'variables_created':len(outputs),
    'sql_file':str(sql_path),
    'status':'PASS'}])
closure_path=LOG_DIR/'stage3_36_busqueda_ayuda_closure.csv'
closure.to_csv(closure_path,index=False)
display(closure)
print('Notebook 08 completed successfully.')
print('Closure log:', closure_path)

,run_utc,table,expected_rows,actual_rows,variables_created,sql_file,status
0,2026-07-17T05:11:40.746047+00:00,enares-2024-crs04.enares2024_crs04_analytical....,18807,18807,37,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,PASS


Notebook 08 completed successfully.
Closure log: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/stage03/stage3_36_busqueda_ayuda_closure.csv
